# Sec. 6.2: final velocity-transformation comparison

Run all cells from this folder (the folder containing `data/`).

The plotting cell below is the user-edited final layout, preserved unchanged. Only the calculations and plotting used in the manuscript figure remain. The `.py` file exports the same functions for command-line use.


In [34]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.integrate import cumulative_simpson, simpson
from scipy.io import loadmat
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter, MultipleLocator, MaxNLocator, AutoMinorLocator
from matplotlib import font_manager

# Run the notebook from the folder containing data/.
HERE = Path.cwd().resolve()

METHODS = ['Raw', 'TL', 'IT-pi', 'AI Buckingham']
PROFILE_COLORS = ['#7B8CDE', '#EAA05D', '#59BF78', '#E96A73', '#AA7BCB', '#AF805E']
RE_B = [9993,15491,23978,9979,8430,18224]
M_B = [1.7,1.7,3.,4.,1.5,1.5]
METHOD_STYLES = {'Raw':('#888888','s'),'TL':('#0072B2','o'),
                 'IT-pi':('#D55E00','D'),'AI Buckingham':('#009E73','^')}
LABELS = {'Raw':'Untransformed','TL':'TL','IT-pi':r'IT-$\pi$','AI Buckingham':'AI Buckingham'}


In [35]:
def style():
    for p in Path('C:/Windows/Fonts').glob('times*.ttf'):
        font_manager.fontManager.addfont(str(p))
    family = 'Times New Roman'
    if not any(f.name == family for f in font_manager.fontManager.ttflist):
        raise RuntimeError('Times New Roman is required for this manuscript figure.')
    plt.rcParams.update({'font.family':family, 'font.size':9, 'mathtext.fontset':'custom',
        'mathtext.rm':family, 'mathtext.it':family+':italic', 'mathtext.bf':family+':bold',
        'mathtext.cal':family+':italic','mathtext.sf':family,'mathtext.tt':family,
        'mathtext.fallback':None,
        'axes.labelsize':10, 'axes.titlesize':10, 'legend.fontsize':7.5,
        'xtick.labelsize':9, 'ytick.labelsize':9, 'axes.linewidth':.7,
        'xtick.direction':'in', 'ytick.direction':'in', 'pdf.fonttype':42,
        'ps.fonttype':42, 'text.usetex':False})
    return family



In [36]:
from contextlib import contextmanager

@contextmanager
def times_math_layout():
    """Use Times PCLT x-height in font units while rendering this figure.

    Matplotlib 3.10 reads these Times metrics at the wrong scale. The local
    correction keeps the font and mathematical notation unchanged, and is
    restored after savefig, including on exceptions.
    """
    from matplotlib import _mathtext
    from matplotlib import text as mtext
    from matplotlib.mathtext import MathTextParser

    original = _mathtext.TruetypeFonts.get_xheight

    def clear_layout_caches():
        MathTextParser._parse_cached.cache_clear()
        mtext._get_text_metrics_with_cache_impl.cache_clear()

    def corrected(self, fontname, fontsize, dpi):
        font = self._get_font(fontname)
        if font.family_name == 'Times New Roman':
            pclt = font.get_sfnt_table('pclt')
            if pclt and pclt.get('xHeight', 0) > 0 and font.units_per_EM > 0:
                return (float(pclt['xHeight']) / font.units_per_EM
                        * fontsize * dpi / 72.0)
        return original(self, fontname, fontsize, dpi)

    clear_layout_caches()
    _mathtext.TruetypeFonts.get_xheight = corrected
    try:
        yield
    finally:
        _mathtext.TruetypeFonts.get_xheight = original
        clear_layout_caches()



In [37]:
def format_axes(ax, *, profile=False):
    ax.set_xscale('log')
    ax.set_xlim(.1 if profile else 1,1000)
    ax.xaxis.set_major_locator(LogLocator(base=10,numticks=5))
    ax.xaxis.set_minor_locator(LogLocator(base=10,subs=np.arange(2,10)*.1,numticks=100))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.grid(which='major',ls='--',lw=.45,color='#D8D8D8',alpha=.75)
    ax.grid(which='minor',ls='--',lw=.3,color='#D8D8D8',alpha=.30)
    ax.tick_params(which='major',direction='in',length=3)
    ax.tick_params(which='minor',direction='in',length=1.6)
    if profile:
        ax.set_ylim(0,30);ax.yaxis.set_major_locator(MultipleLocator(10))
    else:
        ax.set_ylim(bottom=0)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4,min_n_ticks=3))
        ax.yaxis.set_minor_locator(AutoMinorLocator(2))


In [38]:
def calculate(data_dir, subdivisions=64):
    """Compute the three transformed profiles and the errors in the final figure."""
    data_dir = Path(data_dir)
    df = pd.read_csv(data_dir/'dataset_veloTrans.csv', float_precision='round_trip')
    ref = loadmat(data_dir/'incom_ref.mat')
    refx = ref['y_plus'].ravel()
    reference = PchipInterpolator(refx, ref['U_utau'].ravel(), extrapolate=False)
    curves = {}
    native = []
    for case, g in df.groupby('case', sort=True):
        g = g.sort_values('y')
        case = int(case)
        y = (g.y*g.rho_w*g.u_tau/g.mu_w).to_numpy()
        u = g['u+'].to_numpy()
        R = (g.rho/g.rho_w).to_numpy()
        M = (g.mu/g.mu_w).to_numpy()
        assert y[0] == u[0] == 0 and R[0] == M[0] == 1
        assert np.all(np.diff(y) > 0)
        pr, pm, pu = [PchipInterpolator(y, v, extrapolate=False) for v in [R, M, u]]
        dr, dm, du = pr.derivative(), pm.derivative(), pu.derivative()

        def kernel(t):
            r, m = pr(t), pm(t)
            return np.sqrt(r)*(1+t*dr(t)/(2*r)-t*dm(t)/m)

        def integrand(t):
            return kernel(t)*du(t)

        n = subdivisions
        yd = np.r_[np.concatenate([
            np.linspace(s, e, n+1)[:-1] for s, e in zip(y[:-1], y[1:])
        ]), y[-1]]
        rd, md, ud = pr(yd), pm(yd), pu(yd)
        xtl = yd*np.sqrt(rd)/md
        utl = cumulative_simpson(integrand(yd), x=yd, initial=0)
        xai = yd*rd**(2/3)*md**(-1/3)
        uai = ud*rd**.86*md**.40
        xit = yd*rd**(-.06)*md**(-1.38)
        uit = ud*rd**1.44*md**1.23
        assert all(np.all(np.diff(x) > 0) for x in [xtl, xai, xit])
        curves[case] = {
            'Raw': (yd, ud), 'TL': (xtl, utl),
            'IT-pi': (xit, uit), 'AI Buckingham': (xai, uai)
        }
        native.append(pd.DataFrame({
            'case': case, 'y_plus': y, 'U_plus': u, 'R': R, 'M': M,
            'x_TL': y*np.sqrt(R)/M, 'U_TL': utl[::n],
            'x_AI': y*R**(2/3)*M**(-1/3), 'U_AI': u*R**.86*M**.40,
            'x_IT': y*R**(-.06)*M**(-1.38), 'U_IT': u*R**1.44*M**1.23
        }))
    assert len(curves) == 6 and refx[0] == 0
    assert min(refx[-1], *(x[-1] for c in curves.values() for x, _ in c.values())) >= 200
    interpolants = {
        j: {m: PchipInterpolator(x, u, extrapolate=False) for m, (x, u) in c.items()}
        for j, c in curves.items()
    }

    # Panel (b): RMS of six residuals at every common reference coordinate.
    rms_grid = np.geomspace(1., 200., 1201)
    six_rows = []
    for m in METHODS:
        residual = np.array([
            interpolants[j][m](rms_grid)-reference(rms_grid) for j in curves
        ])
        assert residual.shape == (6, len(rms_grid)) and np.isfinite(residual).all()
        values = np.sqrt(np.mean(residual**2, axis=0))
        six_rows.extend({
            'method': m, 'y_plus_reference': float(y), 'profiles': 6, 'RMS_Uplus': float(v)
        } for y, v in zip(rms_grid, values))

    # Panels (c,d): spatial RMS divided by the reference mean velocity.
    normalized_rows = []
    for lo, hi in [(0., 30.), (30., 200.)]:
        y = np.linspace(lo, hi, 20001)
        rr = reference(y)
        reference_mean = float(simpson(rr, x=y)/(hi-lo))
        for j in curves:
            for m in METHODS:
                residual = interpolants[j][m](y)-rr
                assert np.isfinite(residual).all()
                rms = float(np.sqrt(simpson(residual**2, x=y)/(hi-lo)))
                normalized_rows.append({
                    'case': j, 'method': m, 'y_plus_min': lo, 'y_plus_max': hi,
                    'RMS_absolute_Uplus': rms, 'reference_mean_Uplus': reference_mean,
                    'normalized_RMS_percent': 100*rms/reference_mean
                })
    return (pd.concat(native, ignore_index=True), reference,
            pd.DataFrame(six_rows), pd.DataFrame(normalized_rows))


In [39]:
@times_math_layout()
def render_final_comparison(native,reference,six_rms,normalized,output):
    """Two-by-three figure: fixed-six-profile RMSE and regional NRMSE by case."""
    style()
    plt.rcParams.update({'axes.labelsize':14,'xtick.labelsize':11,
                         'ytick.labelsize':11,'legend.fontsize':11.5,
                         'axes.linewidth':.9,'xtick.major.width':.9,
                         'ytick.major.width':.9,'xtick.minor.width':.65,
                         'ytick.minor.width':.65})
    # Decrease legend_y to move both legend blocks towards the first row.
    legend_y = .985
    panel_fontsize = 14.5
    fig=plt.figure(figsize=(8.0,5.))
    gs=fig.add_gridspec(2,3,left=.068,right=.985,bottom=.115,top=.81,
                        height_ratios=[1.12,1],wspace=.3,hspace=.6)
    top=[fig.add_subplot(gs[0,i]) for i in range(3)]
    bottom=[fig.add_subplot(gs[1,i]) for i in range(3)]
    xx=np.geomspace(.1,1000,1200)
    xcols={'TL':'x_TL','IT-pi':'x_IT','AI Buckingham':'x_AI'}
    ucols={'TL':'U_TL','IT-pi':'U_IT','AI Buckingham':'U_AI'}
    for ax,m in zip(top,METHODS[1:]):
        for j,g in native.groupby('case'):
            k=int(j)-1;valid=g[xcols[m]]>0
            ax.plot(g.loc[valid,xcols[m]],g.loc[valid,ucols[m]],'.',
                ms=4.8,alpha=.67,color=PROFILE_COLORS[k],
                label=rf'$Re_b={RE_B[k]},\ M_b={M_B[k]:.1f}$')
        ax.plot(xx,reference(xx),color='#303030',lw=1.15,label='DNS reference')
        format_axes(ax,profile=True)
        ax.set_ylim(-2,30)  # Margin below the near-wall points.
        ax.set_title(LABELS[m],fontsize=14.5,pad=5)
        ax.set_xlabel({'TL':r'$y_{\mathrm{TL}}^+$','IT-pi':r'$\Pi_{\mathrm{IT}}$',
                       'AI Buckingham':r'$\Pi_1^*$'}[m],labelpad=0)
        ax.set_ylabel({'TL':r'$U_{\mathrm{TL}}^+$','IT-pi':r'$U_{\mathrm{IT}}^+$',
                       'AI Buckingham':r'$\tilde{\Pi}_y$'}[m],labelpad=0)
    ax=bottom[0]
    for m in METHODS:
        color,marker=METHOD_STYLES[m]
        t=six_rms[six_rms.method==m]
        ax.plot(t.y_plus_reference,t.RMS_Uplus,color=color,lw=1.3,
            marker=marker,markevery=max(1,len(t)//12),ms=5.2,mfc='white',mew=1.,
            label=LABELS[m])
    ax.set_xscale('log');ax.set_xlim(1,200)
    ax.xaxis.set_major_locator(LogLocator(base=10,numticks=4))
    ax.xaxis.set_minor_locator(LogLocator(base=10,subs=np.arange(2,10)*.1,numticks=100))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel(r'$y^+$',labelpad=0)
    ax.set_ylabel(r'$\bar{e}(y^+)$',labelpad=0)
    ax.set_title('RMS error',fontsize=14.5,pad=5)
    ax.set_yscale('log');ax.set_ylim(1e-2,20)
    for k,ax in enumerate(bottom[1:],start=1):
        lo,hi=(0.,30.) if k==1 else (30.,200.)
        data=normalized[(normalized.y_plus_min==lo)&(normalized.y_plus_max==hi)]
        for mi,m in enumerate(METHODS):
            t=data[data.method==m].sort_values('case');color,marker=METHOD_STYLES[m]
            # Small within-category offsets distinguish methods; x is a case ID,
            # not a continuous physical quantity such as Re_b.
            ax.plot(t.case+(mi-1.5)*.09,t.normalized_RMS_percent,ls='none',color=color,
                    marker=marker,ms=6.2,mfc='white',mew=1.0,label=LABELS[m])
        ax.set_xlim(.55,6.45);ax.set_xticks(range(1,7))
        ax.set_xlabel('Profile',labelpad=1)
        ax.set_yscale('log');ax.set_ylim(.2,100)
        ax.set_title(r'$0\leq y^+\leq30$' if k==1 else r'$30\leq y^+\leq200$',fontsize=14.5,pad=5)
        ax.set_ylabel(r'$E$ (%)')
    for k,ax in enumerate(bottom):
        ax.yaxis.set_major_locator(LogLocator(base=10,numticks=5))
        ax.yaxis.set_minor_locator(LogLocator(base=10,subs=np.arange(2,10)*.1,numticks=100))
        ax.yaxis.set_minor_formatter(NullFormatter())
        ax.tick_params(which='both',direction='in')
        ax.grid(axis='y',which='major',ls='--',lw=.45,color='#D8D8D8',alpha=.75)
        ax.grid(axis='y',which='minor',ls='--',lw=.30,color='#D8D8D8',alpha=.25)
    # Separate corner tick labels and reserve room around axis labels.
    for ax in top+bottom:
        ax.tick_params(axis='x',which='major',pad=5)
        ax.tick_params(axis='y',which='major',pad=5)
        ax.xaxis.labelpad=3
        ax.yaxis.labelpad=2
    ph,pl=top[0].get_legend_handles_labels()
    order=[0,3,1,4,2,5]
    fig.legend([ph[i] for i in order],[pl[i] for i in order],ncol=3,
        loc='upper left',bbox_to_anchor=(.0,legend_y),frameon=False,fontsize=11.5,
        handlelength=.5,columnspacing=.4,handletextpad=.12,markerscale=1.55,
        borderaxespad=0,labelspacing=.3)
    mh,ml=bottom[0].get_legend_handles_labels()
    # Matplotlib fills each legend column vertically: DNS reference below Raw.
    handles=mh+[ph[-1]];labels=ml+[pl[-1]]
    method_order=[0,4,1,3,2]
    fig.legend([handles[i] for i in method_order],
        [labels[i] for i in method_order],ncol=3,loc='upper left',
        bbox_to_anchor=(.605,legend_y),frameon=False,fontsize=11.5,
        columnspacing=.4,handlelength=.8,handletextpad=.25,
        borderaxespad=0,labelspacing=.3)
    # Align each panel letter with the centre of its vertical y-axis label.
    fig.canvas.draw()
    renderer=fig.canvas.get_renderer()

    # 根据 legend 文字位置，精确设置外框
    from matplotlib.transforms import Bbox
    from matplotlib.patches import FancyBboxPatch

    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    # 获取两个 legend 的整体范围
    legend_bounds = Bbox.union([
        lg.get_window_extent(renderer)
        for lg in fig.legends
    ])

    # 获取所有 legend 文字的实际范围
    text_bounds = Bbox.union([
        txt.get_window_extent(renderer)
        for lg in fig.legends
        for txt in lg.get_texts()
    ])

    # 单位：pt（排版点）
    left_pad = 2
    right_pad = 2
    top_pad = 5
    bottom_pad = 1

    # pt 转换为像素
    scale = fig.dpi / 72

    x0 = legend_bounds.x0 - left_pad * scale
    x1 = legend_bounds.x1 + right_pad * scale

    y0 = text_bounds.y0 - bottom_pad * scale
    y1 = text_bounds.y1 + top_pad * scale

    # 转换到 figure 坐标系
    inv = fig.transFigure.inverted()

    fx0, fy0 = inv.transform((x0, y0))
    fx1, fy1 = inv.transform((x1, y1))

    legend_frame = FancyBboxPatch(
        (fx0, fy0),
        fx1 - fx0,
        fy1 - fy0,
        boxstyle='round,pad=0,rounding_size=0.008',
        facecolor='white',
        edgecolor='#DDDDDD',
        linewidth=0.7,
        transform=fig.transFigure,
        zorder=2,
        clip_on=False
    )

    legend_frame.set_gid('shared-legend-frame')
    fig.add_artist(legend_frame)

    # 获取 (b) 的 x 坐标
    box_b = bottom[0].yaxis.label.get_window_extent(renderer)
    x_b = fig.transFigure.inverted().transform(
        ((box_b.x0 + box_b.x1) / 2, 0)
    )[0]

    for ax, letter in zip([top[0]] + bottom, 'abcd'):
        box = ax.yaxis.label.get_window_extent(renderer)

        x = fig.transFigure.inverted().transform(
            ((box.x0 + box.x1) / 2, 0)
        )[0]

        # (a) 与 (b) 水平对齐
        if letter == 'a':
            x = x_b

        y = ax.get_position().y1 + 4 / (72 * fig.get_figheight())

        fig.text(
            x, y, rf'$({letter})$',
            ha='center',
            va='bottom',
            fontsize=panel_fontsize,
            fontweight='normal'
        )

    fig.savefig(output.with_suffix('.pdf'),bbox_inches='tight')
    fig.savefig(output.with_suffix('.png'),dpi=450,bbox_inches='tight')
    plt.close(fig)


In [40]:
def main():
    out = HERE/'results'
    out.mkdir(exist_ok=True)
    native, reference, six_rms, normalized = calculate(HERE/'data')
    native.to_csv(out/'transformed_native_profiles.csv', index=False)
    six_rms.to_csv(out/'six_profile_pointwise_RMS.csv', index=False)
    normalized.to_csv(out/'normalized_RMS_intervals.csv', index=False)
    means = []
    for (lo, hi, method), g in normalized.groupby(['y_plus_min', 'y_plus_max', 'method']):
        means.append({
            'a': lo, 'b': hi, 'method': method,
            'mean_normalized_RMS_percent': float(g.normalized_RMS_percent.mean())
        })
    means = pd.DataFrame(means)
    means.to_csv(out/'normalized_RMS_across_profiles.csv', index=False)
    render_final_comparison(native, reference, six_rms, normalized, out/'TL_comparison')
    print(means.to_string(index=False))
    print('Saved:', out)


In [41]:
main()


   a     b        method  mean_normalized_RMS_percent
 0.0  30.0 AI Buckingham                     1.943426
 0.0  30.0         IT-pi                     5.513791
 0.0  30.0           Raw                     9.618466
 0.0  30.0            TL                     2.883484
30.0 200.0 AI Buckingham                     1.428362
30.0 200.0         IT-pi                     2.809022
30.0 200.0           Raw                    28.900661
30.0 200.0            TL                     2.368287
Saved: Q:\Work2_agent\2602_Dimensional_agent\Packages_\results
